# Freeze Provenance

This notebook shows how a frozen dataset is produced, checked and traced. It runs
the shared `jute-disease freeze target` command on a synthetic fixture, so it needs
no real images and fits no model.

- **Demonstration:** one command writes the manifest and its receipt.
- **Lineage:** every input is hashed into the receipt; a second process reproduces it.
- **Regression:** a changed input changes only its own hash, and a different freeze
  is never replaced silently.

In [1]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd

from jute_disease.config import CURATION_RULES_PATH
from jute_disease.data import load_manifest, make_synthetic_data

work = Path(tempfile.mkdtemp(prefix="freeze-provenance-"))
plan = make_synthetic_data(work / "images", per_class=20, quarantine_count=4)

rules = json.loads(CURATION_RULES_PATH.read_text(encoding="utf-8"))
rules["min_support"] = 2  # the fixture is small; the real rules require more
(work / "rules.json").write_text(json.dumps(rules), encoding="utf-8")
(work / "reviews.json").write_text(
    json.dumps({"version": 1, "decisions": []}), encoding="utf-8"
)
print(
    f"{len(plan.files)} synthetic images; {len(plan.quarantine_files)} are quarantined by rule"
)

126 synthetic images; 4 are quarantined by rule


In [2]:
def freeze(name: str, rules_file: str = "rules.json", *extra: str) -> dict:
    """Run the shared freeze command in a fresh process and return its report."""
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "jute_disease.cli",
            "freeze",
            "target",
            "--image-root",
            str(work / "images"),
            "--rules",
            str(work / rules_file),
            "--reviews",
            str(work / "reviews.json"),
            "--output",
            str(work / name),
            "--workers",
            "2",
            *extra,
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip().splitlines()[-1])
    return json.loads(result.stdout)

## 1. Demonstration

The receipt is written last, so a directory without one is never a completed freeze.

In [3]:
first = freeze("first")
receipt = first["receipt"]
pd.Series(
    {k: receipt[k] for k in ("seed", "images", "groups", "excluded", "quarantined")}
).to_frame("value")

,value
seed,42
images,119
groups,107
excluded,4
quarantined,2


In [4]:
pd.DataFrame(receipt["support"]).T.rename_axis("partition")

,Cercospora Leaf Spot,Dieback,General Damage,Healthy,Mosaic,Stem Rot
partition,,,,,,
test,3,3,3,3,3,3
train,13,14,14,14,14,14
val,3,3,3,3,3,3


## 2. Lineage

Each dependency is a hash of the input that shaped the manifest. A second process
with the same inputs must produce the same receipt and byte-identical files.

In [5]:
pd.Series(receipt["dependencies"]).to_frame("sha256")

,sha256
identity,953de70393690bd95b13a38c826dd805ee67fd5591e95f...
inventory,18d54dea83074bc3ffa62816f452d511cad6d45d904cc8...
preprocessing,68052201165164855b9cec369cb9307ed4cd254f8d52ed...
reviews,4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba...
rules,be0b52379cd2439035595b9313f578f38e0a5c1d81fce1...


In [6]:
second = freeze("second")
same_receipt = second["receipt"] == receipt
same_bytes = all(
    (work / "first/target" / name).read_bytes()
    == (work / "second/target" / name).read_bytes()
    for name in ("manifest.json", "receipt.json")
)
assert same_receipt and same_bytes
print("receipt equal:", same_receipt, "| files byte-identical:", same_bytes)

receipt equal: True | files byte-identical: True


## 3. Regression

Changing a rule changes the `rules` hash and leaves the inventory hash alone. A
different freeze into the same place is refused until it is explicitly forced.

In [7]:
changed = {**rules, "min_support": 3}
(work / "rules_changed.json").write_text(json.dumps(changed), encoding="utf-8")
moved = freeze("moved", "rules_changed.json")["receipt"]["dependencies"]
old = receipt["dependencies"]
pd.DataFrame(
    {"before": old, "after": moved, "changed": {k: old[k] != moved[k] for k in old}}
)

,before,after,changed
identity,953de70393690bd95b13a38c826dd805ee67fd5591e95f...,953de70393690bd95b13a38c826dd805ee67fd5591e95f...,False
inventory,18d54dea83074bc3ffa62816f452d511cad6d45d904cc8...,18d54dea83074bc3ffa62816f452d511cad6d45d904cc8...,False
preprocessing,68052201165164855b9cec369cb9307ed4cd254f8d52ed...,68052201165164855b9cec369cb9307ed4cd254f8d52ed...,False
reviews,4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba...,4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba...,False
rules,be0b52379cd2439035595b9313f578f38e0a5c1d81fce1...,b1752892a62c7a4cd6f2ab04f5543964d2a8d7deefbb83...,True


In [8]:
try:
    freeze("first", "rules.json", "--seed", "7")
except RuntimeError as error:
    print("refused:", str(error).split(" in ")[0], "(use --force to replace)")
else:
    raise AssertionError("a different freeze replaced the completed one")

refused: freeze blocked: A different completed freeze exists (use --force to replace)


## 4. Reading the frozen partitions

Partitions are read only through the manifest; the groups never span partitions.

In [9]:
manifest = load_manifest(work / "first/target")
frame = pd.DataFrame(
    {
        "partition": [e.partition for e in manifest.entries],
        "group": [e.group for e in manifest.entries],
    }
)
leaks = frame.groupby("group")["partition"].nunique()
assert (leaks == 1).all()
print(f"{len(leaks)} groups, none spanning partitions")

107 groups, none spanning partitions
